In [1]:
import re
import pandas as pd
import os

FILES = {
    ("x86",     "o0"): "../data/x86_64_o0.ll",
    ("x86",     "o3"): "../data/x86_64_o3.ll",
    ("aarch64", "o0"): "../data/aarch64_o0.ll",
    ("aarch64", "o3"): "../data/aarch64_o3.ll",
}

OP_SUFFIX = {
    "encrypt":          "encrypt_block_64_128",
    "encrypt_inflight": "encrypt_block_inflight_64_128",
    "decrypt":          "decrypt_block_64_128",
}
SPEC = []
for grp in ["scalar", "sse2", "avx2", "avx512", "neon"]:
    for op, suffix in OP_SUFFIX.items():
        SPEC.append((grp, op, f"{grp}_{suffix}"))

TARGETS = [name for _, _, name in SPEC]

HEAP_RE        = re.compile(r"@(__rust_(?:alloc|realloc|dealloc|alloc_zeroed))\b")
ARRAY_RE       = re.compile(r"alloca\s+\[(\d+)\s+x\s+i8\]")
ALLOC_SIZE_RE  = re.compile(r"@__rust_alloc(?:_zeroed)?\(i64 (\d+)")
REALLOC_SIZE_RE= re.compile(r"@__rust_realloc\(.*i64 (\d+)\s*\)")

def heap_bytes(line):
    m = ALLOC_SIZE_RE.search(line) or REALLOC_SIZE_RE.search(line)
    return int(m.group(1)) if m else 0


In [2]:
def parse_ir(path):
    lines = open(path).read().splitlines()
    found = {}
    raw_rows = []
    pending, i, n = [], 0, len(lines)

    while i < n:
        line = lines[i]; st = line.strip()
        if st == "":
            pending = []
        elif st.startswith(";"):
            pending.append(st.lstrip("; ").rstrip())
        elif line.startswith("define"):
            name = next((c for c in reversed(pending)
                         if not c.startswith("Function Attrs")), "<unknown>")
            body, j = [line], i + 1
            while j < n and not lines[j].startswith("}"):
                body.append(lines[j]); j += 1
            match = next((t for t in TARGETS if t in name), None)
            if match and match not in found:
                allocas = [b.strip() for b in body if "alloca" in b]
                heaps   = [b.strip() for b in body if HEAP_RE.search(b)]
                found[match] = dict(
                    mangled=name,
                    alloca_count=len(allocas),
                    stack_bytes=sum(int(m.group(1)) for b in allocas
                                    if (m := ARRAY_RE.search(b))),
                    heap_count=len(heaps),
                    heap_bytes=sum(heap_bytes(b) for b in heaps),
                )
                for a in allocas:
                    m = ARRAY_RE.search(a)
                    raw_rows.append(dict(func=match, kind="stack",
                                         bytes=int(m.group(1)) if m else None,
                                         ir=a))
                for h in heaps:
                    raw_rows.append(dict(func=match, kind="heap",
                                         bytes=heap_bytes(h) or None,
                                         ir=h))
            pending, i = [], j
        i += 1

    summary = []
    for grp, op, name in SPEC:
        d = found.get(name)
        summary.append(dict(
            group=grp, op=op, func=name, present=d is not None,
            alloca_count=d["alloca_count"] if d else 0,
            stack_bytes =d["stack_bytes"]  if d else 0,
            heap_count  =d["heap_count"]   if d else 0,
            heap_bytes  =d["heap_bytes"]   if d else 0,
            mangled     =d["mangled"]      if d else None,
        ))
    df = pd.DataFrame(summary)
    raw = pd.DataFrame(raw_rows, columns=["func", "kind", "bytes", "ir"])
    return df, raw


In [3]:
dfs, raws = [], []
for (arch, opt), path in FILES.items():
    d, r = parse_ir(path)
    d.insert(0, "arch", arch); d.insert(1, "opt", opt)
    r.insert(0, "arch", arch); r.insert(1, "opt", opt)
    dfs.append(d); raws.append(r)

df  = pd.concat(dfs,  ignore_index=True)
raw = pd.concat(raws, ignore_index=True)

df["arch"]  = pd.Categorical(df["arch"],  categories=["x86","aarch64"], ordered=True)
df["opt"]   = pd.Categorical(df["opt"],   categories=["o0","o3"], ordered=True)
df["group"] = pd.Categorical(df["group"],
                categories=["scalar","sse2","avx2","avx512","neon"], ordered=True)
df["op"]    = pd.Categorical(df["op"],
                categories=["encrypt","encrypt_inflight","decrypt"], ordered=True)
df = df.sort_values(["arch","opt","group","op"]).reset_index(drop=True)


In [4]:
present = df[df.present].drop(columns=["present","mangled"])
present


,arch,opt,group,op,func,alloca_count,stack_bytes,heap_count,heap_bytes
0,x86,o0,scalar,encrypt,scalar_encrypt_block_64_128,8,156,0,0
1,x86,o0,scalar,encrypt_inflight,scalar_encrypt_block_inflight_64_128,7,48,0,0
2,x86,o0,scalar,decrypt,scalar_decrypt_block_64_128,8,156,0,0
3,x86,o0,sse2,encrypt,sse2_encrypt_block_64_128,1251,20464,0,0
4,x86,o0,sse2,encrypt_inflight,sse2_encrypt_block_inflight_64_128,1249,20016,0,0
5,x86,o0,sse2,decrypt,sse2_decrypt_block_64_128,1251,20464,0,0
6,x86,o0,avx2,encrypt,avx2_encrypt_block_64_128,1251,40928,0,0
7,x86,o0,avx2,encrypt_inflight,avx2_encrypt_block_inflight_64_128,1249,40032,0,0
8,x86,o0,avx2,decrypt,avx2_decrypt_block_64_128,1251,40928,0,0
9,x86,o0,avx512,encrypt,avx512_encrypt_block_64_128,721,47936,0,0


In [5]:
present.pivot_table(
    index=["group","op"], columns=["arch","opt"],
    values="stack_bytes", observed=True
)


arch                         x86       aarch64     
opt                           o0   o3       o0   o3
group  op                                          
scalar encrypt             156.0  0.0    156.0  0.0
       encrypt_inflight     48.0  0.0     48.0  0.0
       decrypt             156.0  0.0    156.0  0.0
sse2   encrypt           20464.0  0.0      NaN  NaN
       encrypt_inflight  20016.0  0.0      NaN  NaN
       decrypt           20464.0  0.0      NaN  NaN
avx2   encrypt           40928.0  0.0      NaN  NaN
       encrypt_inflight  40032.0  0.0      NaN  NaN
       decrypt           40928.0  0.0      NaN  NaN
avx512 encrypt           47936.0  0.0      NaN  NaN
       encrypt_inflight  46144.0  0.0      NaN  NaN
       decrypt           47936.0  0.0      NaN  NaN
neon   encrypt               NaN  NaN  20464.0  0.0
       encrypt_inflight      NaN  NaN  20016.0  0.0
       decrypt               NaN  NaN  20464.0  0.0

In [6]:
present.groupby(["arch","opt","group"], observed=True)[
    ["alloca_count","stack_bytes","heap_count","heap_bytes"]].sum()


alloca_count  stack_bytes  heap_count  heap_bytes
arch    opt group                                                    
x86     o0  scalar            23          360           0           0
            sse2            3751        60944           0           0
            avx2            3751       121888           0           0
            avx512          2161       142016           0           0
        o3  scalar             0            0           0           0
            sse2               0            0           0           0
            avx2               0            0           0           0
            avx512             0            0           0           0
aarch64 o0  scalar            23          360           0           0
            neon            3751        60944           0           0
        o3  scalar             0            0           0           0
            neon               0            0           0           0

In [7]:
backend_map = {
    "scalar": "Skalarny",
    "sse2":   "SSE2",
    "avx2":   "AVX2",
    "avx512": "AVX-512",
    "neon":   "NEON",
}

op_map = {
    "encrypt":          "Szyfrowanie",
    "encrypt_inflight": "Szyfr. w locie",
    "decrypt":          "Deszyfrowanie",
}

arch_map = {
    "x86": "x86\_64",
}

<>:16: SyntaxWarning: "\_" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\_"? A raw string is also an option.
<>:16: SyntaxWarning: "\_" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\_"? A raw string is also an option.
/tmp/ipykernel_47855/3940724603.py:16: SyntaxWarning: "\_" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\_"? A raw string is also an option.
  "x86": "x86\_64",


In [8]:
table = (present.groupby(["arch","opt","group","op"], observed=True)[
            ["alloca_count","stack_bytes","heap_count","heap_bytes"]].sum()
         .unstack("opt")
         .swaplevel(axis=1)
         .sort_index(axis=1))

table = table.drop(columns=["heap_count", "heap_bytes"], level=1)

table = table.rename(index=arch_map,    level="arch")
table = table.rename(index=backend_map, level="group")
table = table.rename(index=op_map,      level="op")

os.makedirs("../csv", exist_ok=True)

out = table.copy()
out.columns = ["_".join(map(str, c)) for c in out.columns]
out = out.reset_index()
out.to_csv("../csv/memory_results.csv", index=False)

In [9]:
def print_raw(arch, opt, kinds=("stack", "heap")):
    sub = raw[(raw.arch == arch) & (raw.opt == opt)]
    print("=" * 70)
    print(f" {arch}  {opt}")
    print("=" * 70)
    for grp, op, name in SPEC:
        d = df[(df.arch == arch) & (df.opt == opt) & (df.func == name)]
        if d.empty or not bool(d.present.iloc[0]):
            continue
        r0 = d.iloc[0]
        rows = sub[sub.func == name]
        print(f"\n=== [{grp}/{op}] {r0.mangled}")
        if "stack" in kinds:
            print(f"   stos:   {r0.alloca_count} alloca, >= {r0.stack_bytes} B")
            for ir in rows[rows.kind == "stack"].ir:
                print(f"            {ir}")
        if "heap" in kinds:
            print(f"   sterta: {r0.heap_count} wywolan, >= {r0.heap_bytes} B")
            for ir in rows[rows.kind == "heap"].ir:
                print(f"            {ir}")


In [10]:
print_raw("x86", "o0")


 x86  o0

=== [scalar/encrypt] speck_probe::speck::backend::scalar::encrypt_block::scalar_encrypt_block_64_128
   stos:   8 alloca, >= 156 B
            %y = alloca [4 x i8], align 4
            %x = alloca [4 x i8], align 4
            %k = alloca [4 x i8], align 4
            %l = alloca [12 x i8], align 4
            %round_keys = alloca [108 x i8], align 4
            %_0 = alloca [8 x i8], align 4
            %1 = alloca [8 x i8], align 8
            %ct = alloca [8 x i8], align 4
   sterta: 0 wywolan, >= 0 B

=== [scalar/encrypt_inflight] speck_probe::speck::backend::scalar::encrypt_block::scalar_encrypt_block_inflight_64_128
   stos:   7 alloca, >= 48 B
            %y = alloca [4 x i8], align 4
            %x = alloca [4 x i8], align 4
            %k = alloca [4 x i8], align 4
            %l = alloca [12 x i8], align 4
            %_0 = alloca [8 x i8], align 4
            %1 = alloca [8 x i8], align 8
            %ct = alloca [8 x i8], align 4
   sterta: 0 wywolan, >= 0 B

=== 

In [11]:
print_raw("x86", "o3")


 x86  o3

=== [scalar/encrypt] speck_probe::speck::backend::scalar::encrypt_block::scalar_encrypt_block_64_128
   stos:   0 alloca, >= 0 B
   sterta: 0 wywolan, >= 0 B

=== [scalar/encrypt_inflight] speck_probe::speck::backend::scalar::encrypt_block::scalar_encrypt_block_inflight_64_128
   stos:   0 alloca, >= 0 B
   sterta: 0 wywolan, >= 0 B

=== [scalar/decrypt] speck_probe::speck::backend::scalar::decrypt_block::scalar_decrypt_block_64_128
   stos:   0 alloca, >= 0 B
   sterta: 0 wywolan, >= 0 B

=== [sse2/encrypt] speck_probe::speck::backend::x86_64::sse2::encrypt_block::sse2_encrypt_block_64_128
   stos:   0 alloca, >= 0 B
   sterta: 0 wywolan, >= 0 B

=== [sse2/encrypt_inflight] speck_probe::speck::backend::x86_64::sse2::encrypt_block::sse2_encrypt_block_inflight_64_128
   stos:   0 alloca, >= 0 B
   sterta: 0 wywolan, >= 0 B

=== [sse2/decrypt] speck_probe::speck::backend::x86_64::sse2::decrypt_block::sse2_decrypt_block_64_128
   stos:   0 alloca, >= 0 B
   sterta: 0 wywolan, >=

In [12]:
print_raw("aarch64", "o0")


 aarch64  o0

=== [scalar/encrypt] speck_probe::speck::backend::scalar::encrypt_block::scalar_encrypt_block_64_128
   stos:   8 alloca, >= 156 B
            %y = alloca [4 x i8], align 4
            %x = alloca [4 x i8], align 4
            %k = alloca [4 x i8], align 4
            %l = alloca [12 x i8], align 4
            %round_keys = alloca [108 x i8], align 4
            %_0 = alloca [8 x i8], align 4
            %1 = alloca [8 x i8], align 8
            %ct = alloca [8 x i8], align 4
   sterta: 0 wywolan, >= 0 B

=== [scalar/encrypt_inflight] speck_probe::speck::backend::scalar::encrypt_block::scalar_encrypt_block_inflight_64_128
   stos:   7 alloca, >= 48 B
            %y = alloca [4 x i8], align 4
            %x = alloca [4 x i8], align 4
            %k = alloca [4 x i8], align 4
            %l = alloca [12 x i8], align 4
            %_0 = alloca [8 x i8], align 4
            %1 = alloca [8 x i8], align 8
            %ct = alloca [8 x i8], align 4
   sterta: 0 wywolan, >= 0 B



In [13]:
print_raw("aarch64", "o3")


 aarch64  o3

=== [scalar/encrypt] speck_probe::speck::backend::scalar::encrypt_block::scalar_encrypt_block_64_128
   stos:   0 alloca, >= 0 B
   sterta: 0 wywolan, >= 0 B

=== [scalar/encrypt_inflight] speck_probe::speck::backend::scalar::encrypt_block::scalar_encrypt_block_inflight_64_128
   stos:   0 alloca, >= 0 B
   sterta: 0 wywolan, >= 0 B

=== [scalar/decrypt] speck_probe::speck::backend::scalar::decrypt_block::scalar_decrypt_block_64_128
   stos:   0 alloca, >= 0 B
   sterta: 0 wywolan, >= 0 B

=== [neon/encrypt] speck_probe::speck::backend::aarch64::neon::encrypt_block::neon_encrypt_block_64_128
   stos:   0 alloca, >= 0 B
   sterta: 0 wywolan, >= 0 B

=== [neon/encrypt_inflight] speck_probe::speck::backend::aarch64::neon::encrypt_block::neon_encrypt_block_inflight_64_128
   stos:   0 alloca, >= 0 B
   sterta: 0 wywolan, >= 0 B

=== [neon/decrypt] speck_probe::speck::backend::aarch64::neon::decrypt_block::neon_decrypt_block_64_128
   stos:   0 alloca, >= 0 B
   sterta: 0 wywo